In [0]:
# Databricks notebook source


dbutils.widgets.text("resource_type", "Patient")
resource_type = dbutils.widgets.get("resource_type")

import sys, os
sys.path.append(os.path.abspath("../common"))
from config import raw_path, bronze_table
from pyspark.sql import functions as F

run_date = spark.sql("SELECT date_format(current_date(), 'yyyy-MM-dd') AS d") \
    .collect()[0]["d"]
raw_dir = raw_path(resource_type, run_date)
target_table = bronze_table(resource_type)

# COMMAND ----------
# MAGIC %md ### Read raw Bundles, explode entries, attach metadata
# MAGIC Each raw file is one FHIR `Bundle` page. We `input_file_name()` to
# MAGIC recover `api_url_or_params`/`extraction_timestamp` from the run log
# MAGIC (joined below) rather than re-parsing them out of the JSON.

# COMMAND ----------
def _dir_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

if not _dir_exists(raw_dir):
    print(f"No raw files for {resource_type} on {run_date} — nothing to load into bronze.")
    dbutils.notebook.exit("NO_DATA")

raw_df = (
    spark.read
    .option("multiLine", True)
    .json(raw_dir)
    .withColumn("source_file", F.col("_metadata.file_path"))
)

# Add entry column if it doesn't exist (handles empty FHIR Bundles)
if "entry" not in raw_df.columns:
    raw_df = raw_df.withColumn("entry", F.array())

# Bundle.entry is an array of {resource: {...}}; explode to one row per
# FHIR resource instance.
exploded = (
    raw_df
    .withColumn("entry", F.explode_outer("entry"))
    .select(
        F.col("entry.resource").alias("resource"),
        F.to_json(F.col("entry.resource")).alias("raw_json"),
        "source_file",
    )
    .filter(F.col("resource").isNotNull())
)

bronze_df = (
    exploded
    .withColumn("resource_id", F.col("resource.id"))
    .withColumn("version_id", F.col("resource.meta.versionId"))
    .withColumn("last_updated", F.col("resource.meta.lastUpdated"))
    .withColumn("resource_type", F.lit(resource_type))
    .withColumn("ingestion_date", F.lit(run_date).cast("date"))
    .withColumn("extraction_timestamp", F.current_timestamp())
    .withColumn("api_url_or_params", F.col("source_file"))
    .select(
        "resource_type", "resource_id", "version_id", "last_updated",
        "raw_json", "ingestion_date", "extraction_timestamp",
        "api_url_or_params",
    )
)

# COMMAND ----------
# MAGIC %md ### Write to Bronze Delta table (append; partitioned by ingestion_date)

# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {target_table} (
    resource_type STRING,
    resource_id STRING,
    version_id STRING,
    last_updated STRING,
    raw_json STRING,
    ingestion_date DATE,
    extraction_timestamp TIMESTAMP,
    api_url_or_params STRING
) USING DELTA
PARTITIONED BY (ingestion_date)
""")

(bronze_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(target_table))

row_count = bronze_df.count()
dbutils.jobs.taskValues.set(key="bronze_rows_written", value=row_count)
print(f"[{resource_type}] appended {row_count} rows to {target_table}")
